In [ ]:
# SPDX-FileCopyrightText: 2026 Mario Gemoll
# SPDX-License-Identifier: 0BSD

import os
import subprocess

def is_correct_repo() -> bool:
    try:
        result = subprocess.run(
            ["git", "remote", "get-url", "origin"],
            capture_output=True,
            text=True,
            check=True,
        )
        remote_url = result.stdout.strip()
        return remote_url in [
            "https://github.com/mariogemoll/reinforcement-learning.git",
            "git@github.com:mariogemoll/reinforcement-learning.git",
        ]
    except (subprocess.CalledProcessError, FileNotFoundError):
        return False


if not is_correct_repo():
    !git clone https://github.com/mariogemoll/reinforcement-learning.git

if not os.getcwd().endswith("reinforcement-learning/py"):
    %cd reinforcement-learning/py

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))


In [ ]:
%pip install -q gymnax

In [ ]:
import os

# Must be set before JAX is imported.
# Creates 8 virtual CPU devices so pmap can parallelize across cores.
# Has no effect when a real GPU/TPU is present.
os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=8")

import jax
import jax.numpy as jnp
import gymnax
import optax
import matplotlib.pyplot as plt
from rl.core.policy_gradient import (
    init_mlp_params,
    calculate_returns,
    discounted_returns_to_go
)
from rl.pendulum.policy_gradient import (
    OBS_DIM,
    ACTION_DIM,
    init_policy_params,
    generate_n_rollouts,
    log_probs_per_step,
    reinforce_loss,
    value_loss,
    make_value_training_step,
)

print(f"JAX devices: {jax.devices()}")


In [ ]:
n_seeds = jax.local_device_count()  # 8 virtual CPUs or however many real GPUs
n_iter = 10000
num_rollouts = 32
max_steps = 200
lr_policy = 3e-4
value_lr_mult = 10
gamma = 0.99

In [ ]:
from tqdm.notebook import trange


def plot_multiseed(all_losses, all_returns, title):
    x = jnp.arange(all_returns.shape[1])
    mean_ret = jnp.mean(all_returns, axis=0)
    std_ret = jnp.std(all_returns, axis=0)
    mean_loss = jnp.mean(all_losses, axis=0)
    std_loss = jnp.std(all_losses, axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    for s in range(all_returns.shape[0]):
        axes[0].plot(all_returns[s], alpha=0.6, linewidth=1.2, label=f"seed {s}")
    axes[0].plot(mean_ret, linewidth=2.5, label="mean across seeds", color="black")
    axes[0].fill_between(
        x, mean_ret - std_ret, mean_ret + std_ret,
        alpha=0.2, color="gray", label="\u00b11 std",
    )
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel("Mean episode return")
    axes[0].set_title("Returns")
    axes[0].legend(ncol=2, fontsize=8)

    for s in range(all_losses.shape[0]):
        axes[1].plot(all_losses[s], alpha=0.6, linewidth=1.2, label=f"seed {s}")
    axes[1].plot(mean_loss, linewidth=2.5, label="mean across seeds", color="black")
    axes[1].fill_between(
        x, mean_loss - std_loss, mean_loss + std_loss,
        alpha=0.2, color="gray", label="\u00b11 std",
    )
    axes[1].set_xlabel("Iteration")
    axes[1].set_ylabel("Loss")
    axes[1].set_title("Loss")
    axes[1].legend(ncol=2, fontsize=8)

    fig.suptitle(title, y=1.03)
    plt.tight_layout()
    plt.show()


def run_multiseed(make_carry_fn, training_step_fn, title, chunk_size=100):
    """Run n_seeds independent training runs in parallel via pmap.

    Splits n_iter into chunks so tqdm can show progress. The first chunk
    includes compilation and will appear slower than the rest.
    """
    carries = [make_carry_fn(s) for s in range(n_seeds)]
    stacked = jax.tree.map(lambda *xs: jnp.stack(xs), *carries)
    all_keys = jnp.stack([
        jax.random.split(jax.random.PRNGKey(s * 1000 + 1), n_iter)
        for s in range(n_seeds)
    ])

    n_chunks = n_iter // chunk_size
    key_tail = all_keys.shape[2:]  # () new-style keys, (2,) old-style
    keys_chunked = all_keys.reshape(n_seeds, n_chunks, chunk_size, *key_tail)

    def run_chunk(carry, keys):
        return jax.lax.scan(training_step_fn, carry, keys)

    pmap_chunk = jax.pmap(run_chunk)

    losses_list, returns_list = [], []
    pbar = trange(n_chunks, desc="Compiling...", leave=True)
    for i in pbar:
        stacked, (chunk_losses, chunk_returns) = pmap_chunk(
            stacked, keys_chunked[:, i]
        )
        _ = chunk_returns.block_until_ready()
        losses_list.append(chunk_losses)
        returns_list.append(chunk_returns)
        pbar.set_description("Training")
        pbar.set_postfix(mean_return=f"{float(jnp.mean(chunk_returns)):.1f}")

    all_losses = jnp.concatenate(losses_list, axis=1)
    all_returns = jnp.concatenate(returns_list, axis=1)
    plot_multiseed(all_losses, all_returns, title)


def trajectory_log_probs(params, rollouts):
    """Sum of per-step log probs for each trajectory."""
    max_t = rollouts['rewards'].shape[1]
    mask = jnp.arange(max_t)[None, :] < rollouts['length'][:, None]
    per_step = jax.vmap(log_probs_per_step, in_axes=(None, 0, 0))(
        params, rollouts['observations'], rollouts['actions']
    )
    return jnp.sum(per_step * mask, axis=1)


def make_adam_run(loss_fn):
    """Factory for steps 1-3: single policy network + Adam."""
    env, env_params = gymnax.make("Pendulum-v1")
    opt = optax.chain(
        optax.clip_by_global_norm(1.0), optax.scale_by_adam(), optax.scale(-1.0)
    )

    def make_carry(s):
        params = init_policy_params(
            jax.random.PRNGKey(s), [OBS_DIM, 32, 32, ACTION_DIM]
        )
        return (params, opt.init(params))

    def training_step(carry, key):
        params, opt_state = carry
        rollouts = generate_n_rollouts(
            key, env, env_params, params, num_rollouts, max_steps
        )
        loss, grad = jax.value_and_grad(loss_fn)(params, rollouts)
        updates, new_opt_state = opt.update(grad, opt_state)
        new_params = optax.apply_updates(
            params, jax.tree.map(lambda u: lr_policy * u, updates)
        )
        mask = jnp.arange(max_steps)[None, :] < rollouts['length'][:, None]
        mean_ret = jnp.mean(jnp.sum(rollouts['rewards'] * mask, axis=1))
        return (new_params, new_opt_state), (loss, mean_ret)

    return make_carry, training_step

## Step 1: Vanilla REINFORCE

The simplest policy gradient: multiply each trajectory's total log-probability
by its total return.

$$\mathcal{L} = -\frac{1}{m} \sum_i \log \pi_\theta(\tau_i) \cdot G(\tau_i)$$

Two big problems:

- **High variance**: every action in the trajectory gets the *same* credit
  regardless of when it happened.
- **No baseline**: returns in the range $[-3000, 0]$ all act as
  positive/negative signals, making gradient magnitudes noisy.

In [ ]:
def reinforce_loss_simple(params, rollouts):
    log_probs = trajectory_log_probs(params, rollouts)
    returns = calculate_returns(rollouts)
    return -jnp.mean(log_probs * returns)


run_multiseed(*make_adam_run(reinforce_loss_simple), "Step 1: Vanilla REINFORCE + Adam")

## Step 2: Normalized advantages

Subtract the mean return and divide by the standard deviation before using it as a signal:

$$A_i = \frac{G(\tau_i) - \mu_G}{\sigma_G + \epsilon}$$

This is purely a change to the loss function. It reduces gradient variance and removes the
scale dependence on raw return magnitudes, but it doesn't fix the credit assignment
problem: every step in the trajectory still gets the same advantage.

In [ ]:
def reinforce_normalized_loss(params, rollouts):
    log_probs = trajectory_log_probs(params, rollouts)
    returns = calculate_returns(rollouts)
    advantages = (returns - returns.mean()) / (returns.std() + 1e-8)
    return -jnp.mean(log_probs * advantages)


run_multiseed(
    *make_adam_run(reinforce_normalized_loss), "Step 2: Normalized Advantages + Adam"
)


## Step 3: Per-step discounted returns-to-go

Instead of one return per trajectory, compute $G_t$ for *each step*: the discounted
sum of rewards from that step onward.

$$G_t = \sum_{t'=t}^{T} \gamma^{t'-t} r_{t'}$$

Then the loss becomes a sum over *steps*, not just trajectories:

$$\mathcal{L} = -\frac{1}{N} \sum_i \sum_t
\log \pi_\theta(a_t^i | s_t^i) \cdot \hat{A}_t^i$$

where $\hat{A}_t^i$ is the normalized $G_t$.

This fixes the credit assignment problem: early actions are no longer blamed or praised
for rewards they didn't cause. Still a loss-function-only change, but it requires
computing per-step log probs (not just trajectory totals).

In [ ]:
def reinforce_per_step_loss(params, rollouts):
    max_t = rollouts["rewards"].shape[1]
    mask = jnp.arange(max_t)[None, :] < rollouts["length"][:, None]

    log_probs = jax.vmap(log_probs_per_step, in_axes=(None, 0, 0))(
        params, rollouts["observations"], rollouts["actions"]
    )

    masked_rewards = rollouts["rewards"] * mask
    returns_to_go = jax.vmap(discounted_returns_to_go, in_axes=(0, None))(
        masked_rewards, gamma
    )

    n_valid = jnp.sum(mask)
    mean_a = jnp.sum(returns_to_go * mask) / n_valid
    var_a = jnp.sum((returns_to_go - mean_a) ** 2 * mask) / n_valid
    advantages = (returns_to_go - mean_a) / (jnp.sqrt(var_a) + 1e-8)

    return -jnp.sum(log_probs * advantages * mask) / n_valid


run_multiseed(
    *make_adam_run(reinforce_per_step_loss),
    "Step 3: Per-step Discounted Returns-to-Go + Adam",
)


## Step 4: Ablations: What actually moves the needle?

Starting from step 3 (per-step $G_t$ + Adam), we add two orthogonal improvements
and run each combination separately:

| | Reward scaling | Value network |
|---|---|---|
| **4a** | ✓ | ✗ |
| **4b** | ✗ | ✓ |
| **4c** | ✓ | ✓ |

**Reward scaling**: a running estimate of reward std normalizes the reward signal
each iteration, keeping gradient magnitudes stable across training.

**Value network** $V_\phi(s_t)$: a learned baseline per state,
trained alongside the policy to minimize $(G_t - V_\phi(s_t))^2$.
Reduces advantage variance far more than a global mean.

In [ ]:
def make_ablation_run(reward_scaling, value_network):
    """Factory for step 4 ablations.

    reward_scaling=True  adds a running-std reward normalizer.
    value_network=True   adds a learned V(s) baseline.
    """
    env, env_params = gymnax.make("Pendulum-v1")
    opt = optax.chain(
        optax.clip_by_global_norm(1.0), optax.scale_by_adam(), optax.scale(-1.0)
    )
    vp_lr = lr_policy * value_lr_mult

    if value_network:
        def make_carry(s):
            pp = init_policy_params(
                jax.random.PRNGKey(s * 3), [OBS_DIM, 32, 32, ACTION_DIM]
            )
            vp = init_mlp_params(
                jax.random.PRNGKey(s * 3 + 1), [OBS_DIM, 32, 32, 1]
            )
            return (pp, vp, opt.init(pp), opt.init(vp), jnp.float32(1.0))

        def training_step(carry, key):
            pp, vp, pos, vos, rew_std = carry
            rollouts = generate_n_rollouts(
                key, env, env_params, pp, num_rollouts, max_steps
            )
            if reward_scaling:
                new_rew_std = (
                    0.99 * rew_std + 0.01 * (jnp.std(rollouts["rewards"]) + 1e-8)
                )
                scaled = {**rollouts, "rewards": rollouts["rewards"] / new_rew_std}
            else:
                new_rew_std = rew_std
                scaled = rollouts

            loss, pg = jax.value_and_grad(reinforce_loss, argnums=0)(
                pp, vp, scaled, gamma
            )
            pu, new_pos = opt.update(pg, pos)
            new_pp = optax.apply_updates(
                pp, jax.tree.map(lambda u: lr_policy * u, pu)
            )

            _, vg = jax.value_and_grad(value_loss)(vp, scaled, gamma)
            vu, new_vos = opt.update(vg, vos)
            new_vp = optax.apply_updates(
                vp, jax.tree.map(lambda u: vp_lr * u, vu)
            )

            mask = jnp.arange(max_steps)[None, :] < rollouts["length"][:, None]
            mean_ret = jnp.mean(jnp.sum(rollouts["rewards"] * mask, axis=1))
            return (new_pp, new_vp, new_pos, new_vos, new_rew_std), (loss, mean_ret)

    else:
        def make_carry(s):
            params = init_policy_params(
                jax.random.PRNGKey(s), [OBS_DIM, 32, 32, ACTION_DIM]
            )
            return (params, opt.init(params), jnp.float32(1.0))

        def training_step(carry, key):
            params, opt_state, rew_std = carry
            rollouts = generate_n_rollouts(
                key, env, env_params, params, num_rollouts, max_steps
            )
            if reward_scaling:
                new_rew_std = (
                    0.99 * rew_std + 0.01 * (jnp.std(rollouts["rewards"]) + 1e-8)
                )
                scaled = {**rollouts, "rewards": rollouts["rewards"] / new_rew_std}
            else:
                new_rew_std = rew_std
                scaled = rollouts

            loss, grad = jax.value_and_grad(reinforce_per_step_loss)(params, scaled)
            updates, new_opt_state = opt.update(grad, opt_state)
            new_params = optax.apply_updates(
                params, jax.tree.map(lambda u: lr_policy * u, updates)
            )
            mask = jnp.arange(max_steps)[None, :] < rollouts["length"][:, None]
            mean_ret = jnp.mean(jnp.sum(rollouts["rewards"] * mask, axis=1))
            return (new_params, new_opt_state, new_rew_std), (loss, mean_ret)

    return make_carry, training_step


### 4a: Reward scaling only

Same as step 3 but the rewards are normalized by a running std before computing
$G_t$. No value network, the advantage is still $G_t$ minus a global mean.

In [ ]:
run_multiseed(
    *make_ablation_run(reward_scaling=True, value_network=False),
    "Step 4a: Reward Scaling Only",
)

### 4b: Value network only

Learned $V_\phi(s_t)$ baseline: Advantage is $G_t - V_\phi(s_t)$.
No reward normalization; raw rewards go directly into the loss.

In [ ]:
run_multiseed(
    *make_ablation_run(reward_scaling=False, value_network=True),
    "Step 4b: Value Network Only",
)

### 4c: Both (full model)

Reward scaling + value network baseline

In [ ]:
run_multiseed(
    *make_value_training_step(num_rollouts, max_steps, lr_policy, value_lr_mult, gamma),
    "Step 4c: Reward Scaling + Value Network",
)